***
# Homework 9: SQL and APIs

**Course:** STAT 606 - Computing in Data Science and Statistics SP24

**Name:** Shrivats Sudhir

**NetID:** ssudhir2

**Email:** ssudhir2@wisc.edu

**Collaborators:** Samuel Merten, Amy Merkelz

**Date:** April 8th, 2024
***

In [126]:
import os
import sqlite3

## 1.) Warmup: `sqlite3` (3 points, spent $\approx$ 5 minutes)

**Here is a table similar to the ones that we saw in the lecture slides, describing some information about the colleges in the West Division of the Big 10 conference.**

![](image.png)

**Use `sqlite3` to create a database with a single table (in addition to the standard metainformation tables) called `t_big10west` that recreates the table in the figure above. That is, `t_big10west` should have five columns `(ID, University, City, State, Founded)`, and seven rows corresponding to the seven universities in the table. Save the database in a file called `big10.db`, and include this file in your submission.**

In [127]:
data = [(101, 'University of Illinois', 'Urbana', 'Illinois', 1867),
        (202, 'University of Iowa', 'Iowa City', 'Iowa', 1847),
        (303, 'University of Minnesota', 'Minneapolis', 'Minnesota', 1851),
        (404, 'University of Nebraska', 'Lincoln', 'Nebraska', 1869),
        (505, 'Northwestern University', 'Evanston', 'Illinois', 1851),
        (606, 'Purdue University', 'West Lafayette', 'Indiana', 1869),
        (707 , 'University of Wisconsin', 'Madison', 'Wisconsin', 1849)]

# If big10.db already exists, we'll get yelled at when we
# try to create a DB file now, so delete it if it
# already exists.
UNIV_DB_FILE = 'big10.db'
if os.path.exists( UNIV_DB_FILE ):
    os.remove( UNIV_DB_FILE )

# Create a transaction
conn = sqlite3.connect( UNIV_DB_FILE )
# Create a cursor object
cursor = conn.cursor()

# Create a table
cursor.execute('''
                CREATE TABLE t_big10west ('ID', 
                                          'University', 
                                          'City', 
                                          'State', 
                                          'Founded') 
                ''')

# Insert data into the table
cursor.executemany('''
                    INSERT INTO t_big10west
                    VALUES (?, ?, ?, ?, ?) 
                    ''', data)

# Commit the transaction
conn.commit()

# Close the transaction
conn.close()

In [128]:
# Create a transaction
conn = sqlite3.connect( UNIV_DB_FILE )
# Create a cursor object
cursor = conn.cursor()

for row in cursor.execute(''' 
                          SELECT * FROM t_big10west 
                          '''):
        print(row)

# Close the transaction
conn.close()

(101, 'University of Illinois', 'Urbana', 'Illinois', 1867)
(202, 'University of Iowa', 'Iowa City', 'Iowa', 1847)
(303, 'University of Minnesota', 'Minneapolis', 'Minnesota', 1851)
(404, 'University of Nebraska', 'Lincoln', 'Nebraska', 1869)
(505, 'Northwestern University', 'Evanston', 'Illinois', 1851)
(606, 'Purdue University', 'West Lafayette', 'Indiana', 1869)
(707, 'University of Wisconsin', 'Madison', 'Wisconsin', 1849)


**Oops! There’s a typo in that table. The University of Wisconsin was founded in 1848, not 1849. Write a SQL command to correct the corresponding entry of the table, and save it in a string-valued variable called `big10_correction`. You do not need to run this command (but you probably should, to check that it’s correct, and if you do, and you change the entry in the table in `big10.db`, that’s okay)**

In [129]:
# Create a transaction
conn = sqlite3.connect( UNIV_DB_FILE )
# Create a cursor object
cursor = conn.cursor()

cursor.execute(''' 
               UPDATE t_big10west 
               SET Founded = 1848
               WHERE ID = 707 AND University = 'University of Wisconsin' AND City = 'Madison' AND State = 'Wisconsin'
               ''')

for row in cursor.execute('''
                          SELECT * FROM t_big10west
                          '''):
    print(row)

# Close the transaction
conn.close()

(101, 'University of Illinois', 'Urbana', 'Illinois', 1867)
(202, 'University of Iowa', 'Iowa City', 'Iowa', 1847)
(303, 'University of Minnesota', 'Minneapolis', 'Minnesota', 1851)
(404, 'University of Nebraska', 'Lincoln', 'Nebraska', 1869)
(505, 'Northwestern University', 'Evanston', 'Illinois', 1851)
(606, 'Purdue University', 'West Lafayette', 'Indiana', 1869)
(707, 'University of Wisconsin', 'Madison', 'Wisconsin', 1848)


## 2.) Relational Databases and SQL (7 points, spent $\approx$ 25 minutes)

**In this problem, you’ll interact with a toy SQL database using Python’s built-in sqlite3 package. Documentation can be found at *https://docs.python.org/3/library/sqlite3.html.*** 

**For this problem, we’ll use a popular toy SQLite database, called `Chinook`, which represents a digital music collection. See the documentation at:**

***https://github.com/lerocha/chinook-database/blob/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite***

**or a more detailed explanation. We’ll use the `.sqlite` file `Chinook_Sqlite.sqlite`, which you should download from the GitHub page above.** 

**Note: Don’t forget to save the file in the directory that you’re going to compress and hand in, and make sure that you use a relative path when referring to the file, so that when the grading script runs your code on one of our machines the file path will still work!**

**Load the database using the Python `sqlite3` package. How many tables are in the database? Save the answer in the variable `n_tables`.**

In [130]:
Chinook = 'Chinook_Sqlite.sqlite'
conn = sqlite3.connect( Chinook )
cursor = conn.cursor()

n_tables = 0
for table in cursor.execute('''
                            SELECT *
                            FROM sqlite_master
                            '''):
    if table[0] == 'table':
        n_tables += 1
print(f'There are {n_tables} tables in {Chinook}.')

conn.close()

There are 11 tables in Chinook_Sqlite.sqlite.


**What are the names of the tables in the database? Save the answer as a list of strings, `table_names`.** 

**Note: you should write Python `sqlite3` code to answer this; don’t just look up the answer in the documentation!**

In [131]:
Chinook = 'Chinook_Sqlite.sqlite'
conn = sqlite3.connect( Chinook )
cursor = conn.cursor()

table_names = []
for table in cursor.execute('''
                            SELECT *
                            FROM sqlite_master
                            WHERE type = 'table'
                            '''):
    table_names.append(table[1])
    
print(f'The table names in {Chinook} are:')
print(table_names)

conn.close()

The table names in Chinook_Sqlite.sqlite are:
['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


**Write a function `list_album_ids_by_letter` that takes as an argument a single character (i.e., a string of length one) and returns a list of the primary keys of all the albums whose titles start with that character.** 

**Your function should ignore case, so that the inputs `“a”` and `“A”` yield the same results.** 

**Include error checking that raises an appropriate error in the event that the input is of the wrong type or if it is not a single character.**

In [132]:
def get_column_names(table_name, cursor):
    
    cursor.execute(f'''
                   SELECT * 
                   FROM {table_name}
                   ''')

    cursor.fetchall()

    columns = []
    for col in cursor.description:
        columns.append(col[0])
    
    return columns

In [133]:
def list_album_ids_by_letter(char):

    if not isinstance(char, (str, )):
        raise TypeError(f'char ({char}) must be of type str.')
    
    if len(char) != 1:
        raise ValueError(f'char ({char}) must be a single character.')
    
    char = char.upper()
    
    Chinook = 'Chinook_Sqlite.sqlite'
    conn = sqlite3.connect( Chinook )
    cursor = conn.cursor()

    AlbumID_idx = get_column_names('Album', cursor).index('AlbumId')
    Title_idx = get_column_names('Album', cursor).index('Title')

    result = []
    for table in cursor.execute('''
                                SELECT *
                                FROM Album
                                '''):
        album_title = table[Title_idx]
        if album_title[0] == char:
            result.append(table[AlbumID_idx])
            
    conn.close()

    return result

print(list_album_ids_by_letter('a'))

[10, 14, 15, 24, 26, 29, 74, 75, 85, 89, 90, 94, 95, 96, 120, 139, 160, 167, 168, 169, 203, 224, 232, 233, 248, 254, 272, 273, 285, 296, 307, 319]


**Write a function `list_song_ids_by_album_letter` that takes as an argument a single character and returns a list of the primary keys of all the songs whose album names begin with that letter (again ignoring case).** 

**As in `list_album_ids_by_letter`, your function should ignore case and perform error checking as appropriate.**

**Hint: you’ll need a JOIN statement here. You can use the `cursor.description` attribute to find out about tables and the names of their columns.**

In [134]:
def list_song_ids_by_album_letter(char):

    if not isinstance(char, (str, )):
        raise TypeError(f'char ({char}) must be of type str.')
    
    if len(char) != 1:
        raise ValueError(f'char ({char}) must be a single character.')
    
    char = char.upper()
    album_ids = list_album_ids_by_letter(char)
    
    Chinook = 'Chinook_Sqlite.sqlite'
    conn = sqlite3.connect( Chinook )
    cursor = conn.cursor()

    TrackId_idx = get_column_names('Track', cursor).index('TrackId')
    AlbumId_idx = get_column_names('Track', cursor).index('AlbumId')
    
    result = []
    for table in cursor.execute('''
                                SELECT *
                                FROM Track
                                '''):
        if table[AlbumId_idx] in album_ids:
            result.append(table[TrackId_idx])

    conn.close()

    return result

print(list_song_ids_by_album_letter('a'))

[85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 923, 924, 925, 926, 927, 928, 929, 930, 931, 932, 933, 934, 935, 936, 937, 938, 939, 940, 941, 942, 943, 944, 945, 946, 947, 948, 1073, 1074, 1075, 1076, 1077, 1078, 1079, 1080, 1081, 1082, 1083, 1084, 1085, 1086, 1133, 1134, 1135, 1136, 1137, 1138, 1139, 1140, 1141, 1142, 1143, 1144, 1145, 1146, 1147, 1148, 1149, 1150, 1151, 1152, 1153, 1154, 1155, 1156, 1157, 1201, 1202, 1203, 1204, 1205, 1206, 1207, 1208, 1209, 1210, 1211, 1212, 1213, 1214, 1215, 1216, 1217, 1218, 1219, 1220, 1221, 1222, 1223, 1224, 1225, 1226, 1227, 1228, 1229, 1230, 1231, 1232, 1233, 1234, 1479, 1480, 148

**Write a function `total_cost_by_album_letter` that takes as an argument a single character and returns the total cost of buying all the songs whose album begins with that letter.**

**This cost should be based on the tracks’ unit prices, so that the cost of buying a set of tracks is simply the sum of the unit prices of all the tracks in the set.**

**Again your function should ignore case and perform appropriate error checking.**

In [135]:
def total_cost_by_album_letter(char):

    if not isinstance(char, (str, )):
        raise TypeError(f'char ({char}) must be of type str.')
    
    if len(char) != 1:
        raise ValueError(f'char ({char}) must be a single character.')
    
    char = char.upper()
    song_ids = list_song_ids_by_album_letter(char)

    Chinook = 'Chinook_Sqlite.sqlite'
    conn = sqlite3.connect( Chinook )
    cursor = conn.cursor()

    TrackId_idx = get_column_names('Track', cursor).index('TrackId')
    UnitPrice_idx = get_column_names('Track', cursor).index('UnitPrice')

    summation = 0
    for table in cursor.execute('''
                                SELECT *
                                FROM Track
                                '''):
        if table[TrackId_idx] in song_ids:
            summation += table[UnitPrice_idx]

    conn.close()

    return summation

print(total_cost_by_album_letter('a'))

366.3100000000019


## 3.) Warmup: interacting with the Yelp API (5 points, spent $\approx$ )

**In this problem, you’ll get some practice working with the Yelp API, which we already saw in lecture.**

**First, you need to obtain an API key in order to authenticate to the Yelp API. Follow the instructions at**

***https://docs.developer.yelp.com/docs/fusion-authentication***

**under the section titled “Create an app on Yelp’s Developers site”. You may fill in whatever information you like in the app information.** 

**Note that you will need a Yelp account to create an app, which you need in order to obtain an API key. If you do not feel comfortable doing this, please let me know promptly by email.**

**Once you have filled out your information, you will be given a ClientID and an API Key. This ID and key come with an associated 300 free calls to the Yelp API to use in the month following the day you create your app.** 

**You should not need anywhere near these 300 API calls to test your code, but if you do run out of API calls, please let me know promptly.**